# Merge MediaPipe datasets — publication schema v2

This notebook merges INCLUDE-50, KSL, MINDS-Libras, and LIBRAS-UFOP with **one row per frame** using sequential reads. Run the cells in order from the project root or the `notebooks` directory. Python 3.10+ and pandas are required. The inputs total approximately 12.5 GB, so reserve enough space for another copy of the data.

The output directory contains:

| File | Contents |
| --- | --- |
| `frames.csv` | 12 canonical metadata fields and 1,629 coordinates; 1,641 columns in total. |
| `classes.csv` | One row per class with its label, category, and source identifiers. |
| `samples.csv` | One row per sample with its source file, original identifiers, and segment boundaries. |
| `data_dictionary.csv` | Type, nullability, and meaning of every column in the three CSV files. |
| `README.md` | Standardization rules, keys, missing values, and metadata usage. |
| `manifest.json` | Schema version, inputs, counts, and full/sample export status. |

### Standardization decisions

- `class_id`, `sample_id`, `sequence_id`, and `signer_id` are globally unique IDs prefixed by the dataset name. Aliases such as `class_uid`, `video_uid`, `person`, and `interpreter` are excluded from the main CSV.
- `frame` becomes `frame_id`. The MINDS `category` field identifies the class rather than a thematic category. Labels and categories from different sources are not merged by name or number.
- `sample_id` identifies a video or segment. `sequence_id` groups UFOP segments from the same sequence. For the other sources, which provide no additional grouping, the sequence equals the sample.
- The INCLUDE-50 `split` is preserved; it is empty for the other sources. No signer ID is invented for INCLUDE-50. This merge does not define a joint evaluation protocol.
- Anatomical MINDS landmark names are converted to indices according to `src/preprocessing/landmark_subsets.py` and `src/extraction/mediapipe_extractor.py`.
- Finite coordinate text is preserved without rounding, interpolation, or rescaling. Empty fields and `NaN` use one missing-value representation: an **empty field**. Zero remains a valid value.
- Source `missing_*` flags are replaced by four consistently defined flags: `True` when **any x/y/z coordinate** in the corresponding group is missing. No aggregate `missing_hand` field is exported because its source semantics differ.
- Provenance metadata is stored once per class or sample in the auxiliary tables. Original identifiers document the transformation and are not training aliases.

The export validates frame keys, types, expected columns, metadata consistency, and splits within samples and sequences. Errors stop the export without silently deduplicating or correcting data. Inputs are never modified, and the final directory appears only after the complete export passes validation.


In [1]:
from pathlib import Path
from collections import Counter
from itertools import islice
from urllib.parse import quote
from datetime import datetime, timezone
import csv
import json
import hashlib
import math
import os
import sqlite3
import tempfile

import pandas as pd
from IPython.display import display

def find_project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "data/interim").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Set ROOT manually to the project root.")

ROOT = find_project_root()  # Or: Path("/path/to/islr-subset")
SOURCES = {
    "include50": ROOT / "data/interim/include50/include50_mediapipe_with_split.csv",
    "ksl": ROOT / "data/interim/ksl/ksl_mediapipe.csv",
    "minds": ROOT / "data/interim/minds/minds_mediapipe.csv",
    "ufop": ROOT / "data/interim/ufop/ufop_mediapipe.csv",
}

# None processes every row. Use 100 for a test export, which will be marked as a sample.
MAX_ROWS_PER_DATASET = None
PROGRESS_EVERY = 10_000
output_name = "multilingual-islr-mediapipe" if MAX_ROWS_PER_DATASET is None else "multilingual-islr-mediapipe-sample"
OUTPUT_DIR = ROOT / "data/processed" / output_name
# Existing directories are never overwritten. Choose another OUTPUT_DIR for a new version.
print(f"Output directory: {OUTPUT_DIR}")

Output directory: data/processed/multilingual-islr-mediapipe


## 1. Canonical schema and source-header inspection


In [2]:
HAND_LANDMARKS = ['wrist', 'thumb_cmc', 'thumb_mcp', 'thumb_ip', 'thumb_tip', 'index_finger_mcp', 'index_finger_pip', 'index_finger_dip', 'index_finger_tip', 'middle_finger_mcp', 'middle_finger_pip', 'middle_finger_dip', 'middle_finger_tip', 'ring_finger_mcp', 'ring_finger_pip', 'ring_finger_dip', 'ring_finger_tip', 'pinky_mcp', 'pinky_pip', 'pinky_dip', 'pinky_tip']

POSE_LANDMARKS = ['nose', 'left_eye_inner', 'left_eye', 'left_eye_outer', 'right_eye_inner', 'right_eye', 'right_eye_outer', 'left_ear', 'right_ear', 'mouth_left', 'mouth_right', 'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow', 'left_wrist', 'right_wrist', 'left_pinky', 'right_pinky', 'left_index', 'right_index', 'left_thumb', 'right_thumb', 'left_hip', 'right_hip', 'left_knee', 'right_knee', 'left_ankle', 'right_ankle', 'left_heel', 'right_heel', 'left_foot_index', 'right_foot_index']

LANDMARK_COLUMNS = [
    f"{part}_{index}_{axis}"
    for part, count in (("face", 468), ("hand_0", 21), ("pose", 33), ("hand_1", 21))
    for index in range(count)
    for axis in "xyz"
]
LANDMARK_SET = set(LANDMARK_COLUMNS)
LANDMARK_ALIASES = {
    f"{part}_{name}_{axis}": f"{part}_{index}_{axis}"
    for part, names in (("hand_0", HAND_LANDMARKS), ("hand_1", HAND_LANDMARKS), ("pose", POSE_LANDMARKS))
    for index, name in enumerate(names)
    for axis in "xyz"
}


FRAME_METADATA = [
    "dataset", "class_id", "sample_id", "sequence_id", "signer_id", "frame_id",
    "source_frame_id", "split", "missing_hand_0", "missing_hand_1", "missing_pose", "missing_face",
]
OUTPUT_COLUMNS = FRAME_METADATA + LANDMARK_COLUMNS
CLASS_COLUMNS = [
    "class_id", "source_class_id", "label", "category_name", "source_category_id",
    "source_class_key", "source_class_id_in_category",
]
SAMPLE_COLUMNS = [
    "sample_id", "source_sample_id", "source_sequence_id", "source_signer_id",
    "source_video_name", "source_video_relpath", "source_start_frame", "source_end_frame",
]
GROUP_COLUMNS = {
    part: [column for column in LANDMARK_COLUMNS if column.startswith(part + "_")]
    for part in ("hand_0", "hand_1", "pose", "face")
}
SOURCE_KEYS = {
    "include50": {"class": "sign_id", "frame": "frame_id", "sample": ("sequence_id",), "signer": None},
    "ksl": {"class": "sign_id", "frame": "frame_id", "sample": ("interpreter", "video_name"), "signer": "interpreter"},
    "minds": {"class": "category", "frame": "frame", "sample": ("person", "video_name"), "signer": "person"},
    "ufop": {"class": "sign_id", "frame": "frame_id", "sample": ("sample_id",), "signer": "participant_id"},
}
SOURCE_METADATA = {
    "include50": "sign_id category sign sign_key sample_id video_name video_relpath sequence_id frame_id missing_hand_0 missing_hand_1 missing_hand missing_pose missing_face split".split(),
    "ksl": "sign sign_id interpreter video_name frame_id missing_hand missing_face".split(),
    "minds": "category video_name frame person missing_hand missing_face".split(),
    "ufop": "sign_id sign_key category_id sign_id_in_category participant_id sample_id sequence_id video_name start_frame end_frame frame_id original_frame_id missing_hand missing_face missing_pose".split(),
}

def inspect_sources(sources):
    schemas = {}
    for dataset, path in sources.items():
        with path.open(encoding="utf-8-sig", newline="") as handle:
            original = next(csv.reader(handle), [])
        normalized = [LANDMARK_ALIASES.get(column, column) for column in original]
        duplicates = [column for column, count in Counter(normalized).items() if count > 1]
        if duplicates:
            raise ValueError(f"{dataset}: duplicate columns after standardization: {duplicates}")
        expected = LANDMARK_SET | set(SOURCE_METADATA[dataset])
        if set(normalized) != expected:
            raise ValueError(
                f"{dataset}: unexpected schema; "
                f"missing={sorted(expected - set(normalized))[:10]}, "
                f"extra={sorted(set(normalized) - expected)[:10]}"
            )
        keys = SOURCE_KEYS[dataset]
        required = {keys["class"], keys["frame"], *keys["sample"], "video_name"}
        if keys["signer"]:
            required.add(keys["signer"])
        if dataset == "include50":
            required.add("split")
        if dataset == "ufop":
            required.update(("sequence_id", "start_frame", "end_frame", "original_frame_id"))
        schemas[dataset] = {"original": original, "normalized": normalized, "required": required}
    return schemas

SCHEMAS = inspect_sources(SOURCES)
display(pd.DataFrame([
    {"dataset": dataset, "GB": round(path.stat().st_size / 1e9, 2),
     "source_columns": len(SCHEMAS[dataset]["original"]), "coordinates": len(LANDMARK_COLUMNS)}
    for dataset, path in SOURCES.items()
]))
print(f"frames.csv: {len(OUTPUT_COLUMNS)} columns ({len(FRAME_METADATA)} metadata + {len(LANDMARK_COLUMNS)} coordinates).")

,dataset,GB,source_columns,coordinates
0,include50,1.90,1644,1629
1,ksl,3.39,1636,1629
2,minds,3.46,1635,1629
3,ufop,3.74,1644,1629


frames.csv: 1641 columns (12 metadata + 1629 coordinates).


## 2. Data dictionary

IDs are strings; preserve leading zeros. The `frames.csv` key is (`sample_id`, `frame_id`); `class_id` references `classes.csv`, and `sample_id` references `samples.csv`. The `dataset` field is retained explicitly for source filtering. Repeated IDs across frames are necessary to identify records; competing aliases have been removed.


In [3]:
# field: (logical type, nullable, description)
FRAME_FIELDS = {
    "dataset": ("string", False, "Source dataset: include50, ksl, minds, or ufop."),
    "class_id": ("string", False, "Global class ID: dataset::source ID with percent encoding. References classes.csv."),
    "sample_id": ("string", False, "Global video/segment ID; references samples.csv. It does not identify the signer."),
    "sequence_id": ("string", False, "Groups segments from the same UFOP sequence; equals sample_id for the other sources. Use it to group splits."),
    "signer_id": ("string", True, "Source-prefixed signer ID; empty for INCLUDE-50. It does not link people across datasets."),
    "frame_id": ("integer", False, "Non-negative frame index within the sample, copied from frame_id or frame. It is not renumbered; gaps are preserved."),
    "source_frame_id": ("integer", True, "UFOP original_frame_id in the source convention. Empty for other sources; no shared frame_id base is assumed."),
    "split": ("string", True, "train, val, or test for INCLUDE-50; empty for other datasets. No new split is created."),
    "missing_hand_0": ("boolean", False, "True if any x/y/z coordinate from hand 0 (left in the extraction convention) is missing; otherwise False."),
    "missing_hand_1": ("boolean", False, "True if any x/y/z coordinate from hand 1 (right in the extraction convention) is missing; otherwise False."),
    "missing_pose": ("boolean", False, "True if any pose x/y/z coordinate is missing; otherwise False."),
    "missing_face": ("boolean", False, "True if any face x/y/z coordinate is missing; otherwise False."),
}
CLASS_FIELDS = {
    "class_id": FRAME_FIELDS["class_id"],
    "source_class_id": ("string", False, "Source sign_id, or category for MINDS; leading zeros are preserved."),
    "label": ("string", True, "Source sign text, available for INCLUDE-50 and KSL. It is not translated and is empty when unavailable."),
    "category_name": ("string", True, "INCLUDE-50 thematic category. The MINDS category is not copied into this field."),
    "source_category_id": ("string", True, "UFOP category_id; this is the source code rather than a category name."),
    "source_class_key": ("string", True, "Original INCLUDE-50/UFOP sign_key preserved for provenance."),
    "source_class_id_in_category": ("string", True, "Original UFOP sign_id_in_category preserved for provenance."),
}
SAMPLE_FIELDS = {
    "sample_id": FRAME_FIELDS["sample_id"],
    "source_sample_id": ("string", True, "Original INCLUDE-50/UFOP sample_id. It may not be globally unique and must not be used as the primary key."),
    "source_sequence_id": ("string", True, "Original unprefixed INCLUDE-50/UFOP sequence_id. Empty when the source does not provide it."),
    "source_signer_id": ("string", True, "KSL interpreter, MINDS person, or UFOP participant_id; empty for INCLUDE-50."),
    "source_video_name": ("string", False, "Original video_name, without inferring whether it represents a complete file or a segment."),
    "source_video_relpath": ("string", True, "Original INCLUDE-50 video_relpath; empty for other sources."),
    "source_start_frame": ("integer", True, "Original UFOP start_frame, the segment start in the source convention."),
    "source_end_frame": ("integer", True, "Original UFOP end_frame, the inclusive segment end in the source convention."),
}

def data_dictionary():
    records = []
    for table, fields in (("frames.csv", FRAME_FIELDS), ("classes.csv", CLASS_FIELDS), ("samples.csv", SAMPLE_FIELDS)):
        for column, (dtype, nullable, description) in fields.items():
            records.append({"table": table, "column": column, "type": dtype, "nullable": nullable, "description": description})
        if table == "frames.csv":
            for column in LANDMARK_COLUMNS:
                records.append({
                    "table": table, "column": column, "type": "float", "nullable": True,
                    "description": f"MediaPipe coordinate {column}. Finite numeric text is preserved; an empty field represents a missing value. No rescaling or coordinate-frame conversion is applied.",
                })
    return records

PUBLICATION_README = '# Combined MediaPipe dataset — schema v2\n\n`frames.csv` contains one row per frame and 1,641 columns: 12 canonical metadata fields and 1,629 coordinates (543 landmarks × 3 axes). See `data_dictionary.csv` for the type, nullability, and meaning of every column. CSV files use UTF-8, commas, and one header row. Missing values are empty fields; booleans are `True` or `False`.\n\n## Keys and metadata\n\nThe frame key is (`sample_id`, `frame_id`). The keys of `classes.csv` and `samples.csv` are `class_id` and `sample_id`. These tables store labels and provenance once rather than repeating them in every frame.\n\nGlobal IDs have a dataset prefix. Components are separated by `::` and percent-encoded with `urllib.parse.quote(safe="")`. Classes and people are not aligned across sources. `signer_id` is empty for INCLUDE-50.\n\n`frame_id` preserves the local index. UFOP `source_frame_id`, `source_start_frame`, and `source_end_frame` preserve the source conventions and inclusive boundaries. Do not change their base without checking the source definition.\n\n## Transformations and missing values\n\nAnatomical MINDS hand and pose names are converted to MediaPipe indices. The order is face 0–467, hand 0 0–20, pose 0–32, and hand 1 0–20, with x, y, and z for every index. Finite values are not rounded, interpolated, or rescaled. Empty fields and case-insensitive `NaN` values become empty fields; zero remains valid.\n\n`missing_hand_0`, `missing_hand_1`, `missing_pose`, and `missing_face` are recomputed. `True` means that any coordinate in the group is missing, including partial detections. These flags do not measure confidence or occlusion. The ambiguous aggregate `missing_hand` source field is excluded.\n\n## Splits and validation\n\nThe original INCLUDE-50 train/validation/test split is preserved; other split values are empty. No joint protocol is created. Keep all frames from a `sample_id` and all segments from a `sequence_id` in the same partition. Do not use `sample_id` as a signer proxy.\n\nThe export rejects unexpected headers, non-numeric or infinite coordinates, duplicate frame keys, invalid integers, inconsistent UFOP bounds, and metadata changes within a class, sample, or sequence. It does not discard frames automatically.\n\n`manifest.json` records inputs, counts, and `complete_export`. If `complete_export` is false, the files are only a row-limited test and videos may be incomplete. Counts describe processed inputs and do not establish completeness of the original datasets. Licenses and extraction versions cannot be inferred from the CSV files.\n\n## Reading\n\nRead IDs as strings to preserve leading zeros. Let pandas interpret empty coordinate fields as `NaN`, and read large files in chunks. Join frames to classes by `class_id` and to samples by `sample_id` with `validate="many_to_one"`. Do not join by sign names or source-local IDs.\n'

display(pd.DataFrame(data_dictionary()).query("table == 'frames.csv'").head(len(FRAME_METADATA)))

,table,column,type,nullable,description
0,frames.csv,dataset,string,False,"Source dataset: include50, ksl, minds, or ufop."
1,frames.csv,class_id,string,False,Global class ID: dataset::source ID with perce...
2,frames.csv,sample_id,string,False,Global video/segment ID; references samples.cs...
3,frames.csv,sequence_id,string,False,Groups segments from the same UFOP sequence; e...
4,frames.csv,signer_id,string,True,Source-prefixed signer ID; empty for INCLUDE-5...
5,frames.csv,frame_id,integer,False,"Non-negative frame index within the sample, co..."
6,frames.csv,source_frame_id,integer,True,UFOP original_frame_id in the source conventio...
7,frames.csv,split,string,True,"train, val, or test for INCLUDE-50; empty for ..."
8,frames.csv,missing_hand_0,boolean,False,True if any x/y/z coordinate from hand 0 (left...
9,frames.csv,missing_hand_1,boolean,False,True if any x/y/z coordinate from hand 1 (righ...


## 3. Normalization and export

Rows are written sequentially. Only class and sample metadata remains in memory; frame keys are checked in a temporary SQLite database. Auxiliary CSV files are written at the end, and the staging directory is renamed only after every validation passes.


In [4]:
def make_id(dataset, *parts):
    return "::".join([dataset, *(quote(part, safe="") for part in parts)])

def nonnegative_integer(value, field):
    if not value or not value.isascii() or not value.isdecimal():
        raise ValueError(f"{field}: expected a non-negative integer; received {value!r}.")
    return int(value)

def normalize_row(dataset, row):
    keys = SOURCE_KEYS[dataset]
    source_class = row[keys["class"]]
    source_signer = row[keys["signer"]] if keys["signer"] else ""
    class_id = make_id(dataset, source_class)
    sample_id = make_id(dataset, *(row[column] for column in keys["sample"]))
    sequence_id = make_id(dataset, row["sequence_id"]) if dataset == "ufop" else sample_id
    frame_id = nonnegative_integer(row[keys["frame"]], keys["frame"])
    source_frame = ""
    if dataset == "ufop":
        source_frame = nonnegative_integer(row["original_frame_id"], "original_frame_id")
        start = nonnegative_integer(row["start_frame"], "start_frame")
        end = nonnegative_integer(row["end_frame"], "end_frame")
        if not start <= source_frame <= end:
            raise ValueError("original_frame_id is outside the start_frame/end_frame boundaries.")
    split = row.get("split", "")
    if dataset == "include50" and split not in {"train", "val", "test"}:
        raise ValueError(f"invalid split: {split!r}.")
    coordinates = {}
    for column in LANDMARK_COLUMNS:
        value = row[column]
        if value.strip().lower() in {"", "nan"}:
            coordinates[column] = ""
        else:
            if not math.isfinite(float(value)):
                raise ValueError(f"{column}: non-finite value: {value!r}.")
            coordinates[column] = value
    frame = {
        "dataset": dataset, "class_id": class_id, "sample_id": sample_id,
        "sequence_id": sequence_id,
        "signer_id": make_id(dataset, source_signer) if source_signer else "",
        "frame_id": frame_id, "source_frame_id": source_frame, "split": split,
        **{f"missing_{part}": any(coordinates[column] == "" for column in columns)
           for part, columns in GROUP_COLUMNS.items()},
        **coordinates,
    }
    class_record = {
        "class_id": class_id, "source_class_id": source_class,
        "label": row.get("sign", ""),
        "category_name": row["category"] if dataset == "include50" else "",
        "source_category_id": row.get("category_id", ""),
        "source_class_key": row.get("sign_key", ""),
        "source_class_id_in_category": row.get("sign_id_in_category", ""),
    }
    sample_record = {
        "sample_id": sample_id, "source_sample_id": row.get("sample_id", ""),
        "source_sequence_id": row.get("sequence_id", ""), "source_signer_id": source_signer,
        "source_video_name": row["video_name"], "source_video_relpath": row.get("video_relpath", ""),
        "source_start_frame": row.get("start_frame", ""), "source_end_frame": row.get("end_frame", ""),
    }
    return frame, class_record, sample_record

def register_consistent(records, key, value, entity):
    if key in records and records[key] != value:
        raise ValueError(f"Conflicting metadata for {entity} {key!r}.")
    records.setdefault(key, value)

def write_records(path, columns, records):
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns, lineterminator="\n")
        writer.writeheader()
        writer.writerows(records)

def merge_datasets(sources, output_dir, max_rows=None, progress_every=10_000):
    if max_rows is not None and (isinstance(max_rows, bool) or not isinstance(max_rows, int) or max_rows <= 0):
        raise ValueError("max_rows must be None or a positive integer.")
    output_dir = Path(output_dir)
    if output_dir.exists():
        raise FileExistsError(f"Directory already exists: {output_dir}. Choose another directory for this version.")
    if not sources:
        raise ValueError("No dataset was provided.")
    schemas = inspect_sources(sources)
    output_dir.parent.mkdir(parents=True, exist_ok=True)
    summary, input_manifest = [], []
    classes, samples, sample_metadata, sequences = {}, {}, {}, {}
    with tempfile.TemporaryDirectory(dir=output_dir.parent, prefix=".mediapipe-stage-") as temp:
        staging_root = Path(temp)
        bundle = staging_root / "bundle"
        bundle.mkdir()
        connection = sqlite3.connect(staging_root / "frame_keys.sqlite")
        try:
            connection.execute("CREATE TABLE frames (sample_id TEXT, frame_id INTEGER, PRIMARY KEY(sample_id, frame_id)) WITHOUT ROWID")
            with (bundle / "frames.csv").open("w", encoding="utf-8", newline="") as destination:
                writer = csv.DictWriter(destination, fieldnames=OUTPUT_COLUMNS, lineterminator="\n")
                writer.writeheader()
                for dataset, path in sources.items():
                    schema = schemas[dataset]
                    stat_before = path.stat()
                    class_ids, sample_ids, signer_ids = set(), set(), set()
                    splits, missing_counts = Counter(), Counter()
                    count = 0
                    print(f"Processing {dataset}...", flush=True)
                    with path.open(encoding="utf-8-sig", newline="") as source:
                        reader = csv.reader(source)
                        if next(reader) != schema["original"]:
                            raise ValueError(f"{dataset}: header changed while the file was being read.")
                        rows = reader if max_rows is None else islice(reader, max_rows)
                        for row_number, values in enumerate(rows, start=2):
                            try:
                                if len(values) != len(schema["normalized"]):
                                    raise ValueError("Incorrect number of fields.")
                                row = dict(zip(schema["normalized"], values))
                                empty = [column for column in schema["required"] if not row[column].strip()]
                                if empty:
                                    raise ValueError(f"Empty required metadata: {empty}.")
                                frame, class_record, sample_record = normalize_row(dataset, row)
                                register_consistent(classes, frame["class_id"], class_record, "class")
                                register_consistent(samples, frame["sample_id"], sample_record, "sample")
                                sample_attributes = tuple(frame[column] for column in ("dataset", "class_id", "sequence_id", "signer_id", "split"))
                                register_consistent(sample_metadata, frame["sample_id"], sample_attributes, "sample")
                                register_consistent(sequences, frame["sequence_id"], (frame["signer_id"], frame["split"]), "sequence")
                                connection.execute("INSERT INTO frames VALUES (?, ?)", (frame["sample_id"], frame["frame_id"]))
                            except sqlite3.IntegrityError as error:
                                raise ValueError(f"{dataset}, record {row_number}: duplicate frame within the sample.") from error
                            except (ValueError, OverflowError) as error:
                                raise ValueError(f"{dataset}, registro {row_number}: {error}") from error
                            writer.writerow(frame)
                            count += 1
                            class_ids.add(frame["class_id"])
                            sample_ids.add(frame["sample_id"])
                            if frame["signer_id"]:
                                signer_ids.add(frame["signer_id"])
                            splits[frame["split"] or "no_split"] += 1
                            missing_counts.update({f"missing_{part}": int(frame[f"missing_{part}"]) for part in GROUP_COLUMNS})
                            if progress_every and count % progress_every == 0:
                                print(f"  {dataset}: {count:,} frames", flush=True)
                    if not count:
                        raise ValueError(f"{dataset}: file contains no records.")
                    stat_after = path.stat()
                    if (stat_before.st_size, stat_before.st_mtime_ns) != (stat_after.st_size, stat_after.st_mtime_ns):
                        raise ValueError(f"{dataset}: file changed while it was being read.")
                    connection.commit()
                    summary.append({
                        "dataset": dataset, "frames": count, "samples": len(sample_ids),
                        "classes": len(class_ids), "signers": len(signer_ids),
                        **{split: splits[split] for split in ("train", "val", "test", "no_split")},
                        **missing_counts,
                    })
                    input_manifest.append({"dataset": dataset, "file_name": path.name, "bytes": stat_before.st_size, "frames_read": count})
                    print(f"Completed: {dataset} — {count:,} frames", flush=True)
            expected_frames = sum(item["frames"] for item in summary)
            if connection.execute("SELECT COUNT(*) FROM frames").fetchone()[0] != expected_frames:
                raise ValueError("Frame-key count differs from the number of written frames.")
        finally:
            connection.close()
        write_records(bundle / "classes.csv", CLASS_COLUMNS, classes.values())
        write_records(bundle / "samples.csv", SAMPLE_COLUMNS, samples.values())
        dictionary = data_dictionary()
        write_records(bundle / "data_dictionary.csv", list(dictionary[0]), dictionary)
        publication_card = (ROOT / "data/README.md").read_text(encoding="utf-8")
        (bundle / "README.md").write_text(publication_card, encoding="utf-8")
        manifest = {
            "schema_version": "2.0", "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "complete_export": max_rows is None, "max_rows_per_dataset": max_rows,
            "inputs": input_manifest, "summary": summary,
            "totals": {"frames": expected_frames, "classes": len(classes), "samples": len(samples)},
            "frame_columns": len(OUTPUT_COLUMNS),
            "coordinate_missing_value": "", "missing_flag_rule": "any missing x/y/z coordinate in group",
        }
        (bundle / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        checksum_names = ["frames.csv", "classes.csv", "samples.csv", "data_dictionary.csv", "manifest.json", "README.md"]
        with (bundle / "SHA256SUMS").open("w", encoding="utf-8") as checksum_file:
            for name in checksum_names:
                digest = hashlib.sha256()
                with (bundle / name).open("rb") as source_file:
                    for chunk in iter(lambda: source_file.read(1024 * 1024), b""):
                        digest.update(chunk)
                checksum_file.write(f"{digest.hexdigest()}  {name}\n")
        if output_dir.exists():
            raise FileExistsError(f"Destination directory was created during the export: {output_dir}")
        os.rename(bundle, output_dir)
    return pd.DataFrame(summary)

In [5]:
summary = merge_datasets(SOURCES, OUTPUT_DIR, max_rows=MAX_ROWS_PER_DATASET, progress_every=PROGRESS_EVERY)
display(summary)
print(f"Total: {summary['frames'].sum():,} frames")
print(f"Bundle saved to: {OUTPUT_DIR}")
if MAX_ROWS_PER_DATASET is not None:
    print("TEST MODE: videos may be incomplete. Set MAX_ROWS_PER_DATASET = None to export all rows.")

Processing include50...
  include50: 10,000 frames
  include50: 20,000 frames
  include50: 30,000 frames
  include50: 40,000 frames
  include50: 50,000 frames
  include50: 60,000 frames
Completed: include50 — 61,613 frames
Processing ksl...
  ksl: 10,000 frames
  ksl: 20,000 frames
  ksl: 30,000 frames
  ksl: 40,000 frames
  ksl: 50,000 frames
  ksl: 60,000 frames
  ksl: 70,000 frames
  ksl: 80,000 frames
  ksl: 90,000 frames
  ksl: 100,000 frames
Completed: ksl — 108,373 frames
Processing minds...
  minds: 10,000 frames
  minds: 20,000 frames
  minds: 30,000 frames
  minds: 40,000 frames
  minds: 50,000 frames
  minds: 60,000 frames
  minds: 70,000 frames
  minds: 80,000 frames
  minds: 90,000 frames
  minds: 100,000 frames
Completed: minds — 109,392 frames
Processing ufop...
  ufop: 10,000 frames
  ufop: 20,000 frames
  ufop: 30,000 frames
  ufop: 40,000 frames
  ufop: 50,000 frames
  ufop: 60,000 frames
  ufop: 70,000 frames
  ufop: 80,000 frames
  ufop: 90,000 frames
  ufop: 100,00

,dataset,frames,samples,classes,signers,train,val,test,no_split,missing_hand_0,missing_hand_1,missing_pose,missing_face
0,include50,61613,929,50,0,43082,9164,9367,0,10430,16435,0,22
1,ksl,108373,1229,67,20,0,0,0,108373,59099,43884,0,287
2,minds,109392,800,20,8,0,0,0,109392,69247,45803,0,62
3,ufop,115656,3040,56,5,0,0,0,115656,28256,21412,0,502


Total: 395,034 frames
Bundle saved to: data/processed/multilingual-islr-mediapipe


## 4. Inspect the output and join labels

The preview is intentionally small. IDs and textual metadata are read as strings to preserve leading zeros. Joins are many-to-one and cannot multiply frames. To read all coordinates, use `pd.read_csv(OUTPUT_DIR / "frames.csv", dtype={...}, chunksize=...)`.


In [6]:
preview = pd.read_csv(OUTPUT_DIR / "frames.csv", nrows=5, dtype="string", keep_default_na=False)
class_lookup = pd.read_csv(OUTPUT_DIR / "classes.csv", dtype="string", keep_default_na=False)
sample_lookup = pd.read_csv(OUTPUT_DIR / "samples.csv", dtype="string", keep_default_na=False)
assert preview.columns.tolist() == OUTPUT_COLUMNS
assert class_lookup["class_id"].is_unique
assert sample_lookup["sample_id"].is_unique
assert (summary["frames"] == summary[["train", "val", "test", "no_split"]].sum(axis=1)).all()
view = (
    preview[FRAME_METADATA]
    .merge(class_lookup[["class_id", "label", "category_name"]], on="class_id", how="left", validate="many_to_one")
    .merge(sample_lookup[["sample_id", "source_video_name"]], on="sample_id", how="left", validate="many_to_one")
)
assert len(view) == len(preview)
display(view)

,dataset,class_id,sample_id,sequence_id,signer_id,frame_id,source_frame_id,split,missing_hand_0,missing_hand_1,missing_pose,missing_face,label,category_name,source_video_name
0,include50,include50::0,include50::Adjectives_4of8_Adjectives_79._shor...,include50::Adjectives_4of8_Adjectives_79._shor...,,0,,train,False,False,False,False,short,Adjectives,MVI_5110.MOV
1,include50,include50::0,include50::Adjectives_4of8_Adjectives_79._shor...,include50::Adjectives_4of8_Adjectives_79._shor...,,1,,train,False,False,False,False,short,Adjectives,MVI_5110.MOV
2,include50,include50::0,include50::Adjectives_4of8_Adjectives_79._shor...,include50::Adjectives_4of8_Adjectives_79._shor...,,2,,train,False,False,False,False,short,Adjectives,MVI_5110.MOV
3,include50,include50::0,include50::Adjectives_4of8_Adjectives_79._shor...,include50::Adjectives_4of8_Adjectives_79._shor...,,3,,train,False,False,False,False,short,Adjectives,MVI_5110.MOV
4,include50,include50::0,include50::Adjectives_4of8_Adjectives_79._shor...,include50::Adjectives_4of8_Adjectives_79._shor...,,4,,train,False,False,False,False,short,Adjectives,MVI_5110.MOV
